# Hyperparameter tuning on Colab


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

!pip install optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import optuna
import xgboost as xgb
import lightgbm as lgb
import scipy.stats as stats
from sklearn.model_selection import GroupKFold # determine what type of CV split to do
from sklearn.metrics import mean_absolute_error
from scipy.ndimage import binary_erosion
from skimage.morphology import local_maxima
from skimage.measure import label
from PIL import Image
from pathlib import Path
from lightgbm import early_stopping
%matplotlib inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 34.9 MB/s eta 0:00:00


Load repo files.

In [2]:
# Load files
from google.colab import files
files.upload()

# Install and configure kaggle API
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download staix dataset
!kaggle competitions download -c stai-x-challenge-2026

# Make directory
!mkdir -p data
!unzip -q stai-x-challenge-2026.zip -d data


Saving kaggle.json to kaggle.json
100% 95.0M/95.0M [00:06<00:00, 15.3MB/s]



In [3]:
# Download repo
%cd /content
!rm -rf staix26_seasthemoment
!git clone -b tree-eddy https://github.com/lennardpische/staix26_seasthemoment.git
%cd /content/staix26_seasthemoment

# Add repo scripts to path
sys.path.append("/content/staix26_seasthemoment-src")

/content
Cloning into 'staix26_seasthemoment'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 225 (delta 113), reused 164 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 429.21 KiB | 12.62 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/staix26_seasthemoment


Load all data.

In [4]:
from src.data_loader import (
    find_data_dir,
    load_train_data,
    load_val_data,
    load_train_pngs,
    load_val_pngs
)

# Get data directory
# Update function to be compatible with colab
root = Path("/content/data")

# Get training data and validation covariates
train_all_drugs, train_all_opioids, train_all_stims = load_train_data(root)
val_cov = load_val_data(root)

# Get image data
train_imgs, train_img_names = load_train_pngs(root)
val_imgs, val_img_names = load_val_pngs(root)

Feature engineering pipeline.

In [5]:
import importlib
import src.features
importlib.reload(src.features)
from src.features import create_all_features, create_validation_features_from_train_df

# Run feature engineering pipeline on train and val
all_drugs = create_all_features(
    train_all_drugs,
    train_imgs,
    train_img_names
)
all_opioids = create_all_features(
    train_all_opioids,
    train_imgs,
    train_img_names
)
all_stims = create_all_features(
    train_all_stims,
    train_imgs,
    train_img_names
)

Run on validation covariates.

In [6]:
# Compute a reference training set with text intact
ref = create_all_features(
    train_all_stims,
    train_imgs,
    train_img_names,
    rm_text = False,
    rm_date = False
)

# Validation dataframe with rolling statistics
X_val = create_validation_features_from_train_df(
    train_df = ref,
    val_cov_df = val_cov,
    train_imgs = train_imgs,
    train_img_names = train_img_names,
    val_imgs = val_imgs,
    val_img_names = val_img_names
)

Extract covariates and prediction target.

In [7]:
from src.tuning import return_covs_and_target

X_train_drugs, y_train_drugs = return_covs_and_target(all_drugs)
X_train_opioids, y_train_opioids = return_covs_and_target(all_opioids)
X_train_stims, y_train_stims = return_covs_and_target(all_stims)

Tune hyperparameters using 4-fold grouped CV.

In [8]:
from src.tuning import make_xgb_objective, tune_xgb

# Run experiments to find optimal regressor for each dataset
drug_results = tune_xgb(X_train_drugs, y_train_drugs, num_folds = 4)
opioid_results = tune_xgb(X_train_opioids, y_train_opioids, num_folds = 4)
stim_results = tune_xgb(X_train_stims, y_train_stims, num_folds = 4)

[I 2026-06-10 14:35:35,707] A new study created in memory with name: no-name-c18a4b66-9d9d-4800-9125-49f9349abfd4
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [14:35:38] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-06-10 14:35:41,704] Trial 0 finished with value: 3.1309295410420996 and parameters: {'learning_rate': 0.08609406420526426, 'subsample': 1.0, 'colsample_bytree': 0.5, 'max_depth': 8, 'min_child_weight': 7, 'reg_alpha': 0.006154057095183558, 'reg_lambda': 0.00018762687350300122}. Best is trial 0 with value: 3.13092954104209

Print results.

In [9]:
print(drug_results)
print(opioid_results)
print(stim_results)

({'learning_rate': 0.09706755407549365, 'subsample': 0.5, 'colsample_bytree': 1.0, 'max_depth': 3, 'min_child_weight': 3, 'reg_alpha': 1.5937021004273238e-05, 'reg_lambda': 0.5854857665993195, 'n_estimators': 196}, 2.971112533484204)
({'learning_rate': 0.021443156797638366, 'subsample': 0.8, 'colsample_bytree': 1.0, 'max_depth': 3, 'min_child_weight': 7, 'reg_alpha': 0.0031210149531669803, 'reg_lambda': 0.010209440106132521, 'n_estimators': 379}, 1.362782232167428)
({'learning_rate': 0.009383187606263175, 'subsample': 0.7, 'colsample_bytree': 1.0, 'max_depth': 3, 'min_child_weight': 1, 'reg_alpha': 6.703287763962173e-05, 'reg_lambda': 1.9874822388876459, 'n_estimators': 1215}, 0.6906752298728355)
